In [ ]:
using Combinatorics, ITensors, JLD2, ITensorMPS, Plots, NPZ
include("../../src/apply_gate_as_mpo.jl");
include("../../src/gates_utils.jl");
#Script with operations w/ majoranas
include("../../src/majo_machinery.jl");
#Script with possible initial configurations
include("../../src/initial_circuits.jl");
#Script with potential quantum gates
include("../../src/gates_circuit.jl");

In [ ]:
@show Threads.nthreads()

In [ ]:
#number qubits 
nq = 50
#we want the products in module Bm
m = 1
#obtain majoranas and coefficients
list_majoranas = majorana_products(nq, m)
len_maj = length(list_majoranas);
# Number of points to sample
num_points = 1000
# terms involved in the observable
num_terms = len_maj
#Operators out of list_majoranas
list_maj_op = [ops for (_, ops) in list_majoranas]
#Coefficients out of list_majoranas
coeff_maj_op = [real(coef) for (coef,_) in list_majoranas]
# For the FLO we have random coefficients 
Random.seed!(1);
tc = randn(len_maj); random_coeffs = tc/norm(tc);

In [ ]:
#Function to perform multiplication coeff*op
function make_operator(sites::Vector{Index{Int64}}, op::Symbol, index::Int)
    ITensor(paulis[op], sites[index]', sites[index])
end

#Initialize sites for tensors
sites_maj = siteinds("Qubit", nq)

# Generate paulis out of majoranas
paulis_maj = [MPO([make_operator(sites_maj, list_maj_op[i][j], j)
               for j in 1:nq]) for i in 1:len_maj];

# Include coeffcients to be actual majoranas, not just Paulis
op_maj = paulis_maj .* coeff_maj_op;

# Generate FLO 
op_flo_ = paulis_maj .* coeff_maj_op .* random_coeffs;

# Select just a bunch of terms out of FLO 
Random.seed!(1);
indices = randperm(len_maj)[1:num_terms]
op_flo = [op_flo_[i] for i in indices];


In [ ]:
# dataset name
dataset_name = "B$(m)_$(nq)q_$(num_terms)_obs_terms_$(num_points)N"
# ensure that path exists
ispath("../data/$(dataset_name)") || mkpath("../data/$(dataset_name)")

In [ ]:
#Write indices and coeffs in case you want to reuse them in python

npzwrite("../data/$(dataset_name)/indices_$(nq)q.npy", indices)
npzwrite("../data/$(dataset_name)/coeffs_$(nq)q.npy", random_coeffs[indices])

In [ ]:
# Function to get contractions gate |> state

function get_state!(gates, state::MPS, sites::Vector{Index{Int}}, indices::Vector{Vector{Int}})
    final_state = copy(state)
    
    for (j, gate) in enumerate(gates)
        inds = indices[j]
        it = length(size(gate)) == 2 ?
             ITensor(gate, sites[inds[1]]', sites[inds[1]]) :
             ITensor(gate, sites[inds[1]]', sites[inds[2]]', sites[inds[1]], sites[inds[2]])

        final_state = noprime(apply_mpo_gate(final_state, gate_to_mpo(it), inds));
    end

    return final_state
end

In [ ]:
# Function to group observable in case there is a tradeoff in which increasing χ and reducing #operators is faster

function group_mpos(mpos::Vector{MPO}, size_g::Int; maxbond::Int)
    ng = ceil(Int, length(mpos) / size_g)
    grouped = Vector{MPO}(undef, ng)

    for i in 1:ng
        range = (i-1)*size_g + 1 : min(i*size_g, length(mpos))
        grouped[i] = truncate(sum(mpos[range]), maxdim = maxbond)
    end

    return grouped
end

In [ ]:
#Initialize zero state 
zero_state = fill("0",nq)
MPS_zero = productMPS(sites_maj, zero_state);

#how many points obtained as different angles t
t_list = collect(range(0,10,num_points))
npzwrite("../data/$(dataset_name)/t_list_$(nq)q.npy", t_list)

#Layer in quantum circuit
nlayers = 1

#Maximum bond dimension initial state
#If after given layer χ is greater than maxbond then truncate to χ and get state out of circuit
maxbond = 64

#Types of circuits for initialization
state_generators = [
    random_rotations_with_entanglement_state_circuit,
    simple_extent_state_circuit,
    first_qubit_rotation_state_circuit,
    random_rotations_state_circuit,
    random_fermionic_gaussian_circuit
];

#Initial state after applying geenrator circuit 
initial_state = [state_generators[1](MPS_zero,sites_maj,t, nlayers, maxbond; seed = 42) for t in t_list];

# if random_fermionic_gaussian_circuit 
# ngates = 10
# initial_state = [state_generators[5](MPS_zero,sites_maj,ngates, maxbond; seed = 42) for t in t_list];

#Choose size of each group

# size_g = 1


In [ ]:
# A fuction to compute individual overlaos: easier to track down

function calculate_contributions(op_list::Vector{MPO}, st::MPS)

    contributions = [noprime(op * st) for op in op_list];

    return contributions

end

In [ ]:
# Select observable that is full observable (up to chosen number of terms)

observable_flo = op_flo;

# Select observable that is the baiss elements in the module

observable_maj = op_maj;

In [ ]:
# Compute outputs
contributions_mps_flo = Vector{Any}(undef, num_points)
final_state_flo = Vector{Any}(undef, num_points)
outputs = zeros(Float64, num_points)

Threads.@threads for i in 1:num_points
    #println("Processing sample $i on thread $(Threads.threadid())")
    contributions_mps_flo[i] = calculate_contributions(observable_flo, initial_state[i])
    final_state_flo[i] = sum(contributions_mps_flo[i])
    outputs[i] = real(inner(initial_state[i], final_state_flo[i]))
end

In [ ]:
# Compute feature vectors

feat_vectors = Vector{Vector{Float64}}(undef, num_points)

Threads.@threads for i in 1:num_points
    #println("Processing sample $i on thread $(Threads.threadid())")
    vec = Vector{Float64}(undef, len_maj)
    for j in 1:len_maj
        vec[j] = real(inner(initial_state[i], observable_maj[j], initial_state[i]))
    end
    feat_vectors[i] = vec
end

In [ ]:
npzwrite("../data/$(dataset_name)/outputs_$(nq)q.npy", outputs)
array_data = reduce(vcat, feat_vectors')  # convert to 10×N matrix
npzwrite("../data/$(dataset_name)/feat_vec_$(nq)q.npy", array_data)


In [ ]:
# compute kernel matrix
K = zeros(Float64, num_points, num_points)
for i in 1:num_points
    for j in i:num_points
        K[i, j] = dot(feat_vectors[i], feat_vectors[j])
        K[j, i] = K[i, j]
    end
end 

npzwrite("../data/$(dataset_name)/kernel_exact_$(nq)q.npy", K)

In [ ]:
# For reference, compute all pairwise overlaps exactly via MPS
fidelity_K = ones(Float64, num_points, num_points) 
Threads.@threads for i in 1:num_points
    for j in (i + 1):num_points  # skip diagonal, always 1
        fidelity_K[i, j] = real(inner(initial_state[i], initial_state[j]))
        fidelity_K[j, i] = fidelity_K[i, j]
    end
end
npzwrite("../data/$(dataset_name)/fidelity_exact_$(nq)q.npy", fidelity_K)